# Stage 8 — Model Selection

The candidate models are regression models only. A dummy regressor is included as a baseline but is never selected as the final substantive model.

## 8.1 Prepare model inputs

In [374]:
X_train = train_model_data[kept_feature_columns]
y_train = train_model_data["log_u5mr_2023"]
X_test = test_model_data[kept_feature_columns]
y_test = test_model_data["log_u5mr_2023"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (109, 25)
X_test: (37, 25)


## 8.2 Define model pipelines

The model catalogue contains linear baselines, distance-based regression, tree ensembles, Gradient Boosting, and XGBoost when the package is available. XGBoost is optional so the notebook can still run in a standard scikit-learn environment.

In [376]:
def scaled_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model)
    ])


def tree_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model)
    ])

model_catalogue = {
    "Dummy mean": tree_pipeline(DummyRegressor(strategy="mean")),
    "Ridge": scaled_pipeline(Ridge(random_state=RANDOM_STATE)),
    "ElasticNet": scaled_pipeline(ElasticNet(max_iter=10000, random_state=RANDOM_STATE)),
    "KNN": scaled_pipeline(KNeighborsRegressor()),
    "Random Forest": tree_pipeline(RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    "Extra Trees": tree_pipeline(ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    "Gradient Boosting": tree_pipeline(GradientBoostingRegressor(random_state=RANDOM_STATE))
}

if XGBRegressor is not None:
    model_catalogue["XGBoost"] = tree_pipeline(
        XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
else:
    print("XGBoost is not installed. The XGBoost model will be skipped.")

list(model_catalogue.keys())

['Dummy mean',
 'Ridge',
 'ElasticNet',
 'KNN',
 'Random Forest',
 'Extra Trees',
 'Gradient Boosting',
 'XGBoost']